**Timbre Transfer**

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
from features.feature_extraction import extract_features

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataChunk
import numpy as np
import glob

In [8]:
class DDSPDataset(Dataset):
    def __init__(self, data_dir, sr=16000, frame_rate=250, seq_len_sec=4):
        self.sr = sr
        self.frame_rate = frame_rate
        self.seq_len_frames = int(frame_rate * seq_len_sec)
        self.seq_len_samples = int(sr * seq_len_sec)
        
        self.hop_length = self.sr // self.frame_rate
        
        self.audio_files = glob.glob(os.path.join(data_dir, '*.wav'))
        
        if not self.audio_files:
            raise FileNotFoundError(f"Не знайдено аудіофайлів у теці: {data_dir}")

        self.cache = {}

    def _extract_features(self, audio_path):
        f0, loudness, onset, audio = extract_features(audio_path)

        T_frames = min(len(f0), len(loudness), len(onset))
        
        T_samples = T_frames * self.hop_length
        audio = audio[:T_samples]
        data = {
            'f0': torch.tensor(f0, dtype=torch.float32).unsqueeze(-1),
            'loudness': torch.tensor(loudness, dtype=torch.float32).unsqueeze(-1),
            'onset': torch.tensor(onset, dtype=torch.float32).unsqueeze(-1),
            'audio': torch.tensor(audio, dtype=torch.float32),
            'length_frames': T_frames,
            'length_samples': T_samples
        }
        return data

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        audio_path = self.audio_files[idx]
        
        if audio_path not in self.cache:
            self.cache[audio_path] = self._extract_features(audio_path)
        
        data = self.cache[audio_path]
        T_frames = data['length_frames']

        if T_frames < self.seq_len_frames:
            return data['f0'], data['loudness'], data['onset'], data['audio']

        start_frame = np.random.randint(0, T_frames - self.seq_len_frames)
        end_frame = start_frame + self.seq_len_frames
    
        start_sample = start_frame * self.hop_length
        end_sample = end_frame * self.hop_length

        f0_slice = data['f0'][start_frame:end_frame]
        loudness_slice = data['loudness'][start_frame:end_frame]
        onset_slice = data['onset'][start_frame:end_frame]
        audio_slice = data['audio'][start_sample:end_sample]

        return f0_slice, loudness_slice, onset_slice, audio_slice

In [ ]:
def ddsp_collate_fn(batch):
    
    f0s = [item[0] for item in batch]
    loudnesses = [item[1] for item in batch]
    onsets = [item[2] for item in batch]
    audios = [item[3] for item in batch]
    
    f0_batch = torch.stack(f0s, dim=0)
    loudness_batch = torch.stack(loudnesses, dim=0)
    onset_batch = torch.stack(onsets, dim=0)
    audio_batch = torch.stack(audios, dim=0)


    audio_batch = audio_batch.unsqueeze(-1) 

    return f0_batch, loudness_batch, onset_batch, audio_batch

In [ ]:
from torch.utils.data import DataLoader

DATA_DIR = 'data/nsynth-test/audio'
BATCH_SIZE = 4
SR = 16000
FRAME_RATE = 250
SEQ_LEN_SEC = 4

train_dataset = DDSPDataset(
    data_dir=DATA_DIR, 
    sr=SR, 
    frame_rate=FRAME_RATE, 
    seq_len_sec=SEQ_LEN_SEC
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,         
    num_workers=4,        
    collate_fn=ddsp_collate_fn,
    pin_memory=True
)